# Part 1 - Advanced Modelling

This notebook focuses on creating richer Catalyst models, mostly through the `@reaction_network` DSL.


## Part 1.1 - DSL basics

A model is declared inside `@reaction_network begin ... end`. Catalyst then builds a symbolic `ReactionSystem`.


In [ ]:
using Catalyst, Latexify, OrdinaryDiffEq, Plots

rn = @reaction_network begin
    k_X, X --> Y
    k_Y, Y --> X
end

species(rn), parameters(rn), reactions(rn)


In [ ]:
latexify(rn; form = :ode)


In [ ]:
u0 = [:X => 1.0, :Y => 1.0]
tspan = (0.0, 5.0)
ps = [:k_X => 1.0, :k_Y => 2.5]

sol = solve(ODEProblem(rn, u0, tspan, ps), Tsit5())
plot(sol; title = "Two-state conversion model")


## Part 1.2 - Reaction notation patterns

We can represent production/degradation, binding/unbinding, stoichiometric coefficients, and bundled reactions.


In [ ]:
rn_prod_deg = @reaction_network begin
    (p, d), 0 <--> X
end

rn_binding = @reaction_network begin
    (kB, kD), X + Y <--> XY
end

rn_dimer = @reaction_network begin
    (kF, kR), 2X <--> X2
end

rn_bundle = @reaction_network begin
    (dX, dY), (X, Y) --> 0
end

rn_coupled_bundle = @reaction_network begin
    (kX, kY), (X1, Y1) --> (X2, Y2)
end

(rn_prod_deg, rn_binding, rn_dimer, rn_bundle, rn_coupled_bundle)


## Part 1.3 - Non-constant rates and advanced DSL options

Rates can depend on species, parameters, and custom functions.


In [ ]:
my_function(k1, k2, E) = (k1 + E) / (k2 + E)

rn_regulated = @reaction_network begin
    (p, d), 0 <--> E
    hill(E, v, K, n), 0 --> X
    my_function(k1, k2, E), X_i --> X_a
end

latexify(rn_regulated; form = :ode)


If Catalyst cannot infer that a symbol in a rate law is a species, we can explicitly declare it with `@species`.


In [ ]:
rn_with_species_decl = @reaction_network begin
    @species E(t) = 1.0
    E, X_i --> X_a
end

species(rn_with_species_decl), parameters(rn_with_species_decl)


In [ ]:
u0 = [:X_i => 1.0, :X_a => 0.0]
sol = solve(ODEProblem(rn_with_species_decl, u0, (0.0, 10.0)), Tsit5())
plot(sol; title = "Model with explicit species declaration")
